**© Copyright AIDENTIFY. All rights reserved.**

본 자료는 **멀티캠퍼스 LLM 파인튜닝 과정** 수강생을 위해 제작되었으며, 강의 목적으로만 사용 가능합니다.  
무단 복제, 배포, 상업적 이용을 금지합니다.

---

# Session 23: Tool Calling (Function Calling) 개념

## 🎯 Tool Calling (Function Calling) 개요

**Tool Calling**(도구 호출)은 LLM이 외부 도구(함수, API)를 호출하여 실시간 정보를 가져오거나 작업을 수행할 수 있게 하는 기능입니다.

### 왜 Tool Calling이 필요한가?

| LLM의 한계 | Tool Calling 해결책 |
|-----------|--------------------|
| 실시간 정보 없음 | 날씨 API, 뉴스 API 호출 |
| 정확한 계산 불가 | 계산기 도구 호출 |
| 외부 시스템 접근 불가 | DB 조회, 이메일 발송 등 |
| 최신 데이터 부족 | 검색 엔진 호출 |

### Tool Calling 파이프라인

```
사용자 질문 → LLM 판단 → 도구 선택 → 도구 실행 → 결과 반환 → LLM 응답 생성
```

### 학습 목표

- ✅ Tool Calling의 개념과 동작 원리 이해
- ✅ OpenAI Function Calling API 사용법
- ✅ 도구 정의 (JSON Schema) 작성법
- ✅ 실제 도구 구현 및 멀티 도구 호출

### 실습 환경

- 🔧 GPU 불필요 (API 기반 실습)
- 🔧 OpenAI API 키 필요 (gpt-4o-mini 사용)

## 1️⃣ OpenAI Function Calling API

In [ ]:
# 필수 라이브러리
import os
import json
from openai import OpenAI

# API 키 설정
# 방법 1: 환경변수 (권장)
# os.environ["OPENAI_API_KEY"] = "your-api-key-here"

# 방법 2: 직접 입력
client = OpenAI()  # OPENAI_API_KEY 환경변수에서 자동 로드

MODEL = "gpt-4o-mini"  # 비용 효율적 모델

print("✅ OpenAI 클라이언트 초기화 완료")
print(f"📌 모델: {MODEL}")

In [ ]:
# 기본 Function Calling 예시
print("📊 기본 Function Calling 예시")
print("="*60)

# 도구 정의
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "주어진 도시의 현재 날씨 정보를 가져옵니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "날씨를 조회할 도시 이름 (예: 서울, 부산)"
                    }
                },
                "required": ["city"]
            }
        }
    }
]

# 모델에게 질문
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "서울 날씨 어때?"}],
    tools=tools,
    tool_choice="auto"  # 모델이 도구 사용 여부를 자동 판단
)

message = response.choices[0].message
print(f"🔹 사용자: 서울 날씨 어때?")
print(f"\n🔹 모델 응답:")
print(f"   content: {message.content}")
print(f"   tool_calls: {message.tool_calls}")

if message.tool_calls:
    tool_call = message.tool_calls[0]
    print(f"\n📌 도구 호출 감지!")
    print(f"   함수 이름: {tool_call.function.name}")
    print(f"   인자: {tool_call.function.arguments}")

## 2️⃣ 도구 정의 (JSON Schema)

도구는 **JSON Schema** 형식으로 정의합니다. 모델은 이 스키마를 보고 어떤 도구를 호출할지, 어떤 인자를 넘길지 결정합니다.

In [ ]:
# 다양한 도구 정의 예시
print("📊 도구 정의 JSON Schema 예시")
print("="*60)

tools_definitions = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "주어진 도시의 현재 날씨 정보를 가져옵니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "도시 이름 (예: 서울, 부산, 제주)"
                    }
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "수학 표현식을 계산합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "계산할 수학 표현식 (예: 2 + 3 * 4)"
                    }
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_web",
            "description": "웹에서 정보를 검색합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "검색 쿼리"
                    }
                },
                "required": ["query"]
            }
        }
    }
]

print("정의된 도구 목록:")
for tool in tools_definitions:
    func = tool["function"]
    params = list(func["parameters"]["properties"].keys())
    print(f"\n  🔧 {func['name']}")
    print(f"     설명: {func['description']}")
    print(f"     파라미터: {params}")

In [ ]:
# JSON Schema 상세 구조 설명
print("\n📋 JSON Schema 구조 설명")
print("="*60)

example_schema = {
    "type": "function",                     # 고정값
    "function": {
        "name": "함수_이름",                  # 모델이 호출할 함수명
        "description": "함수 설명 (중요!)",   # 모델이 도구 선택에 사용
        "parameters": {                      # 함수 파라미터 정의
            "type": "object",                # 고정값
            "properties": {                  # 각 파라미터 정의
                "param1": {
                    "type": "string",         # string, number, boolean, array 등
                    "description": "파라미터 설명"
                },
                "param2": {
                    "type": "number",
                    "description": "숫자 파라미터"
                }
            },
            "required": ["param1"]           # 필수 파라미터 목록
        }
    }
}

print(json.dumps(example_schema, indent=2, ensure_ascii=False))
print("\n📌 'description'이 매우 중요합니다 - 모델이 이를 보고 도구를 선택합니다!")

## 3️⃣ 실습: 날씨, 계산기, 검색 도구 구현

실제로 동작하는 도구 함수를 구현하고 Tool Calling 파이프라인을 완성합니다.

In [ ]:
# 도구 함수 구현 (실제 API 대신 시뮬레이션)
import random

def get_weather(city: str) -> dict:
    """날씨 API 시뮬레이션"""
    weathers = ["맑음", "흐림", "비", "눈", "구름 많음"]
    result = {
        "city": city,
        "temperature": random.randint(0, 35),
        "condition": random.choice(weathers),
        "humidity": random.randint(30, 90)
    }
    print(f"  🌤️ [도구 실행] get_weather(city='{city}') → {result}")
    return result

def calculate(expression: str) -> dict:
    """계산기 도구"""
    try:
        result = eval(expression)  # 실습용 (보안상 eval은 실무에서 주의)
        print(f"  🔢 [도구 실행] calculate(expression='{expression}') → {result}")
        return {"expression": expression, "result": result}
    except Exception as e:
        return {"error": str(e)}

def search_web(query: str) -> dict:
    """검색 시뮬레이션"""
    results = {
        "query": query,
        "results": [
            {"title": f"{query} 관련 최신 정보", "snippet": f"{query}에 대한 검색 결과입니다."},
            {"title": f"{query} 가이드", "snippet": f"{query}를 이해하기 위한 종합 가이드입니다."}
        ]
    }
    print(f"  🔍 [도구 실행] search_web(query='{query}')")
    return results

# 도구 이름 → 함수 매핑
available_functions = {
    "get_weather": get_weather,
    "calculate": calculate,
    "search_web": search_web
}

print("✅ 도구 함수 3개 정의 완료")
print(f"📌 사용 가능한 도구: {list(available_functions.keys())}")

In [ ]:
# 완전한 Tool Calling 파이프라인 함수
def run_tool_calling(user_message, tools, available_functions, model=MODEL):
    """Tool Calling 전체 파이프라인 실행"""
    print(f"\n{'='*60}")
    print(f"👤 사용자: {user_message}")
    
    messages = [{"role": "user", "content": user_message}]
    
    # Step 1: 모델에게 질문 (도구 사용 여부 판단)
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )
    
    assistant_message = response.choices[0].message
    
    # Step 2: 도구 호출이 필요한지 확인
    if assistant_message.tool_calls:
        print(f"\n🤖 모델 판단: 도구 호출 필요!")
        messages.append(assistant_message)
        
        # Step 3: 각 도구 실행
        for tool_call in assistant_message.tool_calls:
            func_name = tool_call.function.name
            func_args = json.loads(tool_call.function.arguments)
            
            print(f"\n  📞 도구 호출: {func_name}({func_args})")
            
            # 도구 실행
            if func_name in available_functions:
                result = available_functions[func_name](**func_args)
            else:
                result = {"error": f"Unknown function: {func_name}"}
            
            # 도구 결과를 메시지에 추가
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(result, ensure_ascii=False)
            })
        
        # Step 4: 도구 결과로 최종 응답 생성
        final_response = client.chat.completions.create(
            model=model,
            messages=messages
        )
        final_answer = final_response.choices[0].message.content
    else:
        print(f"\n🤖 모델 판단: 도구 필요 없음 (직접 응답)")
        final_answer = assistant_message.content
    
    print(f"\n🤖 최종 응답: {final_answer}")
    return final_answer

print("✅ Tool Calling 파이프라인 함수 정의 완료")

In [ ]:
# 도구가 필요한 질문들 테스트
print("📊 Tool Calling 테스트")
print("="*60)

test_questions = [
    "서울 날씨 알려줘",
    "1234 곱하기 5678은?",
    "LLM 파인튜닝에 대해 검색해줘",
    "안녕하세요! 반갑습니다."  # 도구 불필요한 질문
]

for q in test_questions:
    run_tool_calling(q, tools_definitions, available_functions)

## 4️⃣ 멀티 도구 호출

모델이 하나의 요청에서 **여러 도구를 동시에 호출**할 수 있습니다.

In [ ]:
# 멀티 도구 호출 테스트
print("📊 멀티 도구 호출 테스트")
print("="*60)

multi_tool_questions = [
    "서울이랑 부산 날씨 비교해줘",
    "100달러가 환율 1350원일 때 얼마인지 계산하고, 서울 날씨도 알려줘"
]

for q in multi_tool_questions:
    run_tool_calling(q, tools_definitions, available_functions)

## 5️⃣ Tool Calling 파이프라인 (선택 → 실행 → 응답)

Tool Calling의 전체 흐름을 다이어그램으로 이해합니다.

In [ ]:
# Tool Calling 파이프라인 시각화 (텍스트)
print("📋 Tool Calling 파이프라인")
print("="*60)

pipeline = """
┌──────────────────────────────────────────────────┐
│  1. 사용자 질문                                    │
│     "서울 날씨 알려줘"                              │
└──────────────┬───────────────────────────────────┘
               ▼
┌──────────────────────────────────────────────────┐
│  2. LLM 판단 (도구 선택)                           │
│     - 사용 가능한 도구 목록 확인                      │
│     - 질문에 적합한 도구 선택                         │
│     - 도구 인자(arguments) 추출                     │
│     → get_weather(city="서울")                     │
└──────────────┬───────────────────────────────────┘
               ▼
┌──────────────────────────────────────────────────┐
│  3. 도구 실행                                      │
│     - 실제 함수/API 호출                            │
│     - 결과: {"temperature": 15, "condition": "맑음"} │
└──────────────┬───────────────────────────────────┘
               ▼
┌──────────────────────────────────────────────────┐
│  4. LLM 최종 응답 생성                              │
│     - 도구 결과를 자연어로 변환                        │
│     → "서울은 현재 15도이고 맑습니다."                 │
└──────────────────────────────────────────────────┘
"""

print(pipeline)

In [ ]:
# 도구 사용 여부 판단 기준
print("📋 모델의 도구 선택 기준")
print("="*60)

decision_examples = [
    ("서울 날씨 알려줘", "✅ get_weather", "실시간 정보 필요"),
    ("123 * 456은?", "✅ calculate", "정확한 계산 필요"),
    ("최신 AI 뉴스 찾아줘", "✅ search_web", "외부 정보 검색 필요"),
    ("안녕하세요!", "❌ 도구 불필요", "일반 대화"),
    ("파이썬이 뭐야?", "❌ 도구 불필요", "내장 지식으로 답변 가능"),
    ("서울과 부산 날씨 비교", "✅ get_weather x2", "멀티 도구 호출"),
]

print(f"\n{'질문':<30} {'도구 선택':<20} {'이유':<25}")
print("-"*75)
for q, tool, reason in decision_examples:
    print(f"{q:<30} {tool:<20} {reason:<25}")

## 6️⃣ 실제 응용 사례

In [ ]:
# 실전 도구 세트 확장
print("📊 실전 Tool Calling 시나리오")
print("="*60)

# 실전용 도구 정의
advanced_tools = [
    {
        "type": "function",
        "function": {
            "name": "get_stock_price",
            "description": "주식 종목의 현재 가격을 조회합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "ticker": {"type": "string", "description": "종목 코드 (예: 005930, AAPL)"}
                },
                "required": ["ticker"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "send_email",
            "description": "이메일을 발송합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "to": {"type": "string", "description": "수신자"},
                    "subject": {"type": "string", "description": "제목"},
                    "body": {"type": "string", "description": "본문"}
                },
                "required": ["to", "subject", "body"]
            }
        }
    }
]

# 시뮬레이션 함수
def get_stock_price(ticker):
    prices = {"005930": 72500, "AAPL": 195.50, "NVDA": 875.28}
    price = prices.get(ticker, random.randint(10000, 100000))
    result = {"ticker": ticker, "price": price, "change": f"+{random.uniform(-3, 5):.1f}%"}
    print(f"  💹 [도구 실행] get_stock_price(ticker='{ticker}') → {result}")
    return result

def send_email(to, subject, body):
    result = {"status": "sent", "to": to, "subject": subject}
    print(f"  📧 [도구 실행] send_email(to='{to}', subject='{subject}')")
    return result

# 모든 도구 통합
all_tools = tools_definitions + advanced_tools
all_functions = {
    **available_functions,
    "get_stock_price": get_stock_price,
    "send_email": send_email
}

print(f"✅ 총 {len(all_tools)}개 도구 정의 완료")
for tool in all_tools:
    print(f"  🔧 {tool['function']['name']}: {tool['function']['description'][:50]}")

In [ ]:
# 실전 시나리오 테스트
practical_questions = [
    "삼성전자 주가 알려줘",
    "김과장에게 회의 일정 변경 이메일 보내줘. 내일 3시로 변경됐어.",
    "서울 날씨 확인하고, 100달러가 환율 1350원일 때 얼마인지도 계산해줘",
    "고마워! 도움이 많이 됐어."  # 도구 불필요
]

for q in practical_questions:
    run_tool_calling(q, all_tools, all_functions)

In [ ]:
# Tool Calling 데이터 형식 미리보기 (다음 세션 연결)
print("\n📋 Tool Calling 학습 데이터 형식 미리보기")
print("="*60)

sample_training_data = {
    "messages": [
        {"role": "system", "content": "당신은 도구를 활용할 수 있는 AI 어시스턴트입니다."},
        {"role": "user", "content": "서울 날씨 알려줘"},
        {"role": "assistant", "content": None, "tool_calls": [
            {"type": "function", "function": {"name": "get_weather", "arguments": '{"city": "서울"}'}}
        ]},
        {"role": "tool", "content": '{"temperature": 15, "condition": "맑음"}'},
        {"role": "assistant", "content": "서울은 현재 15도이고 맑은 날씨입니다."}
    ]
}

print(json.dumps(sample_training_data, indent=2, ensure_ascii=False))
print("\n📌 다음 세션에서 이 형식의 학습 데이터를 대량 생성합니다!")

## 📝 정리 및 핵심 요약

### 이번 실습에서 배운 내용

| 항목 | 내용 |
|------|------|
| Tool Calling | LLM이 외부 도구를 호출하여 정보를 가져오는 기능 |
| JSON Schema | 도구를 정의하는 표준 형식 |
| 파이프라인 | 질문 → 도구 선택 → 실행 → 응답 생성 |
| 멀티 도구 | 하나의 요청에서 여러 도구 동시 호출 가능 |

### 핵심 포인트

- 🎯 **Tool Calling은 LLM의 한계를 극복**하는 핵심 기술입니다
- 🎯 **도구 description이 매우 중요** - 모델이 이를 보고 도구를 선택합니다
- 🎯 **도구 불필요한 질문**에는 도구를 호출하지 않는 것도 중요합니다
- 🎯 **멀티 도구 호출**로 복잡한 요청도 처리 가능합니다
- 🎯 OpenAI API 외에 **로컬 sLLM도 파인튜닝으로 Tool Calling 가능**합니다

### 다음 단계

- ➡️ **Notebook 23**: Tool Calling 학습 데이터 생성 - GPT-4o-mini로 합성 데이터 생성
- ➡️ **Notebook 24**: Tool Calling 파인튜닝 - 로컬 sLLM에 Tool Calling 능력 부여